In [1]:
import pandas as pd
import numpy as np

# Brent
oil = pd.read_csv(r"C:\Users\ALIENWARE\OneDrive\Desktop\26s2\ECMT3150\group project\Brent_cleaned.csv", parse_dates=["Date"])

# Airlines
qantas = pd.read_csv(r"C:\Users\ALIENWARE\OneDrive\Desktop\26s2\ECMT3150\group project\Qantas_cleaned.csv", parse_dates=["Date"])
united = pd.read_csv(r"C:\Users\ALIENWARE\OneDrive\Desktop\26s2\ECMT3150\group project\United_cleaned.csv", parse_dates=["Date"])
singapore = pd.read_csv(r"C:\Users\ALIENWARE\OneDrive\Desktop\26s2\ECMT3150\group project\Singapore_cleaned.csv", parse_dates=["Date"])

# Market indices
asx = pd.read_csv(r"C:\Users\ALIENWARE\OneDrive\Desktop\26s2\ECMT3150\group project\ASX200_cleaned.csv", parse_dates=["Date"])
sp500 = pd.read_csv(r"C:\Users\ALIENWARE\OneDrive\Desktop\26s2\ECMT3150\group project\SP500_cleaned.csv", parse_dates=["Date"])
sti = pd.read_csv(r"C:\Users\ALIENWARE\OneDrive\Desktop\26s2\ECMT3150\group project\STI_cleaned.csv", parse_dates=["Date"])

In [2]:
print(oil.columns.tolist())
print(qantas.columns.tolist())
print(asx.columns.tolist())

['Date', 'Brent_Price', 'Oil_Return', 'D', 'Oil_Interaction']
['Date', 'Adjusted_Close', 'Airline_Return']
['Date', 'Close', 'Market_Return']


In [3]:
# Qantas+Brent+ASX200
qantas_merged = (
    qantas[["Date", "Airline_Return"]]
    .merge(
        oil[["Date", "Oil_Return", "D", "Oil_Interaction"]],
        on="Date",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        asx[["Date", "Market_Return"]],
        on="Date",
        how="inner",
        validate="one_to_one"
    )
)

print(qantas_merged.head())
print("Qantas observations:", len(qantas_merged))

        Date  Airline_Return  Oil_Return    D  Oil_Interaction  Market_Return
0 2016-01-04             NaN         NaN  NaN              NaN      -0.480768
1 2016-01-05       -0.305344   -2.004521  0.0              0.0      -1.647114
2 2016-01-06       -0.921666   -4.810143  0.0              0.0      -1.189435
3 2016-01-07        1.227009   -0.948717  0.0              0.0      -2.226399
4 2016-01-08       -0.611623   -5.826295  0.0              0.0      -0.389958
Qantas observations: 2527


In [4]:
# United+Brent+SP500
united_merged = (
    united[["Date", "Airline_Return"]]
    .merge(
        oil[["Date", "Oil_Return", "D", "Oil_Interaction"]],
        on="Date",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        sp500[["Date", "Market_Return"]],
        on="Date",
        how="inner",
        validate="one_to_one"
    )
)

print(united_merged.head())
print("United observations:", len(united_merged))

        Date  Airline_Return  Oil_Return    D  Oil_Interaction  Market_Return
0 2016-01-04       -2.993756         NaN  NaN              NaN      -1.542204
1 2016-01-05       -0.993953   -2.004521  0.0              0.0       0.201020
2 2016-01-06        0.253944   -4.810143  0.0              0.0      -1.320216
3 2016-01-07       -4.767665   -0.948717  0.0              0.0      -2.398582
4 2016-01-08       -1.416024   -5.826295  0.0              0.0      -1.089754
United observations: 2514


In [5]:
# Singapore+Brent+STI
singapore_merged = (
    singapore[["Date", "Airline_Return"]]
    .merge(
        oil[["Date", "Oil_Return", "D", "Oil_Interaction"]],
        on="Date",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        sti[["Date", "Market_Return"]],
        on="Date",
        how="inner",
        validate="one_to_one"
    )
)

print(singapore_merged.head())
print("Singapore observations:", len(singapore_merged))

        Date  Airline_Return  Oil_Return    D  Oil_Interaction  Market_Return
0 2016-01-04       -1.892798         NaN  NaN              NaN      -1.635374
1 2016-01-05        1.175944   -2.004521  0.0              0.0      -0.061373
2 2016-01-06       -1.723410   -4.810143  0.0              0.0      -1.062703
3 2016-01-07       -0.734601   -0.948717  0.0              0.0      -2.687465
4 2016-01-08        1.463871   -5.826295  0.0              0.0       0.777947
Singapore observations: 2511


In [6]:
# Remove duplicates
datasets = {
    "Qantas": qantas_merged,
    "United": united_merged,
    "Singapore": singapore_merged
}

variables = [
    "Airline_Return",
    "Oil_Return",
    "D",
    "Oil_Interaction",
    "Market_Return"
]

for name, df in datasets.items():

    print(f"\n{name}: Before cleaning = {len(df)}")

    df = df.replace([np.inf, -np.inf], np.nan)

    df = df.dropna(subset=variables).copy()

    df = df.sort_values("Date").reset_index(drop=True)

    datasets[name] = df

    print(f"{name}: After cleaning = {len(df)}")


Qantas: Before cleaning = 2527
Qantas: After cleaning = 2451

United: Before cleaning = 2514
United: After cleaning = 2426

Singapore: Before cleaning = 2511
Singapore: After cleaning = 2419


In [7]:
# Verify merged datasets
for name, df in datasets.items():

    print(f"\n===== {name} =====")

    print("Observations:", len(df))

    print("Date range:",
          df["Date"].min(),
          df["Date"].max())

    print("Missing values:")
    print(df.isna().sum())

    print("Duplicate dates:",
          df["Date"].duplicated().sum())

    print("Interaction check:",
          np.allclose(
              df["Oil_Interaction"],
              df["Oil_Return"] * df["D"]
          ))


===== Qantas =====
Observations: 2451
Date range: 2016-01-05 00:00:00 2025-12-31 00:00:00
Missing values:
Date               0
Airline_Return     0
Oil_Return         0
D                  0
Oil_Interaction    0
Market_Return      0
dtype: int64
Duplicate dates: 0
Interaction check: True

===== United =====
Observations: 2426
Date range: 2016-01-05 00:00:00 2025-12-31 00:00:00
Missing values:
Date               0
Airline_Return     0
Oil_Return         0
D                  0
Oil_Interaction    0
Market_Return      0
dtype: int64
Duplicate dates: 0
Interaction check: True

===== Singapore =====
Observations: 2419
Date range: 2016-01-05 00:00:00 2025-12-31 00:00:00
Missing values:
Date               0
Airline_Return     0
Oil_Return         0
D                  0
Oil_Interaction    0
Market_Return      0
dtype: int64
Duplicate dates: 0
Interaction check: True


In [8]:
# Export merged datasets
for name, df in datasets.items():

    df.to_csv(
        f"{name}_regression_data.csv",
        index=False
    )

    print(f"{name}_regression_data.csv saved")

Qantas_regression_data.csv saved
United_regression_data.csv saved
Singapore_regression_data.csv saved


In [9]:
summary = []

for name, df in datasets.items():

    summary.append({
        "Airline": name,
        "Observations": len(df),
        "Start Date": df["Date"].min().date(),
        "End Date": df["Date"].max().date(),
        "Missing Values": df[variables].isna().sum().sum()
    })

summary_df = pd.DataFrame(summary)

display(summary_df)

,Airline,Observations,Start Date,End Date,Missing Values
0,Qantas,2451,2016-01-05,2025-12-31,0
1,United,2426,2016-01-05,2025-12-31,0
2,Singapore,2419,2016-01-05,2025-12-31,0
